In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
from probeinterface import write_probeinterface, read_probeinterface

import os
import numpy as np

import warnings
warnings.filterwarnings('ignore')

import pandas as pd

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
files = sorted(os.listdir("/media/ubuntu/sda/data/mouse6/ns4/natural_image"))

In [3]:
recording_list = []
for file in files:
    recording_raw = se.read_blackrock(file_path=f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/{file}')
    recording_recorded = recording_raw.remove_channels(["98", '31', '32'])
    recording_list.append(recording_recorded)


In [4]:
probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
recording_recorded = si.concatenate_recordings(recording_list)
recording_recorded = recording_recorded.set_probegroup(probe_30channel)

recording_cmr = recording_recorded
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_f, freq=60)

print(recording_f)
recording_cmr = spre.common_reference(recording_f, reference="global", operator="median")
recording_cmr = recording_cmr.rename_channels(['A-000', 'A-001', 'A-002', 'A-003', 'A-004',
                               'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
                               'A-0010', 'A-011', 'A-012', 'A-013', 'A-014',
                               'A-015', 'A-016', 'A-017', 'A-018', 'A-019',
                               'A-020', 'A-021', 'A-022', 'A-023', 'A-024',
                               'A-025', 'A-026', 'A-027', 'A-028', 'A-029'])
print(recording_cmr)

BandpassFilterRecording: 30 channels - 10000.0Hz - 1 segments - 437,380,586 samples 
                         43,738.06s (12.15 hours) - int16 dtype - 24.44 GiB
ChannelSliceRecording: 30 channels - 10000.0Hz - 1 segments - 437,380,586 samples 
                       43,738.06s (12.15 hours) - int16 dtype - 24.44 GiB


In [5]:
output_folder = '/media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full'
recording_preprocessed = recording_cmr.save(format="binary", n_jobs = 20)

default_params = {
        'detect_sign': -1,  
        'adjacency_radius': 120, 
        'freq_min': 300,  
        'freq_max': 3000,
        'filter': True,
        'whiten': True,  
        'num_workers': 20,
        'clip_size': 50,
        'detect_threshold': 5,
        'detect_interval': 3,  
    }
sorting_mountainsort = ss.run_sorter(sorter_name='mountainsort4',
                                recording=recording_preprocessed,
                                remove_existing_folder='True',
                                folder=output_folder,
                                **default_params)

analyzer_mountainsort = si.create_sorting_analyzer(
    sorting=sorting_mountainsort, 
    recording=recording_preprocessed, 
    format='binary_folder', 
    folder=output_folder + '/analyzer_kilosort4_binary'
)

# 计算扩展信息
extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "noise_levels",
    "templates",
    "unit_locations",
    "spike_locations",
    "correlograms",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "spike_locations": {"ms_before": 0.1},
    "correlograms": {"bin_ms": 0.1},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_mountainsort.compute(extensions_to_compute, extension_params=extension_params, n_jobs = 20)

# 读取spikes.npy并检查无效的spike
spikes_path = output_folder + "/analyzer_kilosort4_binary/sorting/spikes.npy"
spikes = np.load(spikes_path)

# 获取recording的总样本数
total_samples = recording_f.get_num_samples()

# 检查第一个和最后一个spike
first_spike_valid = spikes[0]['sample_index'] >= 0
last_spike_valid = spikes[-1]['sample_index'] < total_samples

# 如果第一个或最后一个spike无效，删除所有无效的spike
if not first_spike_valid or not last_spike_valid:
    # 创建有效spike的掩码：sample_index >= 0 且 < total_samples
    valid_mask = (spikes['sample_index'] >= 0) & (spikes['sample_index'] < total_samples)
    spikes_filtered = spikes[valid_mask]
    
    # 保存过滤后的spikes
    np.save(spikes_path, spikes_filtered)
    print(f"删除了 {len(spikes) - len(spikes_filtered)} 个无效的spike")
    print(f"原始spike数量: {len(spikes)}, 过滤后: {len(spikes_filtered)}")
else:
    print("所有spike都在有效范围内")

qm_params = sqm.get_default_qm_params()
analyzer_mountainsort.compute("quality_metrics", qm_params, n_jobs = 20)

# 导出到phy格式
import spikeinterface.exporters as sexp
sexp.export_to_phy(analyzer_mountainsort, output_folder + "/phy_folder_for_kilosort", verbose=True, n_jobs = 20)

Use cache_folder=/tmp/spikeinterface_cache/tmpu4xvk3gx/07B28J0B
write_binary_recording 
engine=process - n_jobs=20 - samples_per_chunk=10,000 - chunk_memory=585.94 KiB - total_memory=11.44 MiB - chunk_duration=1.00s


write_binary_recording (workers: 20 processes): 100%|██████████| 43739/43739 [07:11<00:00, 101.25it/s]


Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording


estimate_sparsity (no parallelization): 100%|██████████| 43739/43739 [00:01<00:00, 25146.84it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/basesorting.py:380: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
noise_level (workers: 20 processes): 100%|██████████| 20/20 [00:00<00:00, 307.46it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/sortinganalyzer.py:2656: DeprecationWarning: The method 'cosine_similarity' is deprecated and will be removed in the next version. Use 'cosine' instead.
  params = self._set_params(**params)
Compute : spike_locations (workers: 20 processes): 100%|██████████| 43739/43739 [00:11<00:00, 3759.63it/s]


所有spike都在有效范围内


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:914: UserWarning: compute_amplitude_cv_metrics() need 'spike_amplitudes' or 'amplitude_scalings'
  warnings.warn("compute_amplitude_cv_metrics() need 'spike_amplitudes' or 'amplitude_scalings'")
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/curation/curation_model.py:355: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.12/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode="after")
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:1658: UserWarning: The `sd_ratio` metric require the `spike_amplitudes` waveform extension. Use the `postprocessing.compute_spike_amplitudes(

Run:
phy template-gui  /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full/phy_folder_for_kilosort/params.py
